# Week 8 Exercise: The Price is Right — IbrahimSheriff

## Overview
This notebook implements a complete Week 8–style pipeline:
- **SpecialistAgent**: Fine-tuned LLM deployed on Modal
- **FrontierAgent**: RAG + frontier model (GPT) with ChromaDB
- **ChromaDB**: Vector store of product embeddings (SentenceTransformer)
- **Ensemble**: Combined predictor (Specialist + Frontier)
- **Evaluation**: Official `evaluate()` on test set (average absolute error)
- **DealAgentFramework + Gradio UI**: Autonomous deal-hunting and table UI

## Prerequisites
- Modal token set; HuggingFace secret in Modal (`huggingface-secret`)
- Deploy pricer: from **week8** folder run `uv run modal deploy -m pricer_service`
- `.env`: `HF_TOKEN`, `OPENAI_API_KEY` (for FrontierAgent)
---

## 1. Setup: path and environment

In [ ]:
import os
import sys
import logging
from pathlib import Path
from dotenv import load_dotenv

load_dotenv(override=True)
os.environ.setdefault("PYTHONIOENCODING", "utf-8")

notebook_dir = Path.cwd()
week8_root = notebook_dir.parent.parent
if str(week8_root) not in sys.path:
    sys.path.insert(0, str(week8_root))

logging.getLogger().setLevel(logging.INFO)
print(f"Week 8 root: {week8_root}")

In [ ]:
required = ["HF_TOKEN", "OPENAI_API_KEY"]
for var in required:
    status = "SET" if os.getenv(var) else "MISSING"
    print(f"  {var}: {status}")

## 2. Load data (Item.from_hub)

In [ ]:
from agents.items import Item
from huggingface_hub import login

hf_token = os.environ.get("HF_TOKEN")
if hf_token:
    login(token=hf_token, add_to_git_credential=False)

LITE_MODE = False
username = "ed-donner"
dataset = f"{username}/items_lite" if LITE_MODE else f"{username}/items_full"

train, val, test = Item.from_hub(dataset)
print(f"Loaded {len(train):,} train, {len(val):,} val, {len(test):,} test items")

## 3. ChromaDB vector store

We build the vector store in the current folder so DealAgentFramework can use it later. Use a subset for speed if you prefer.

In [ ]:
import chromadb
from sentence_transformers import SentenceTransformer
from tqdm.notebook import tqdm

DB = "products_vectorstore"
client = chromadb.PersistentClient(path=DB)
collection = client.get_or_create_collection("products")

def description(item):
    return (item.summary or item.title or "").strip() or getattr(item, "prompt", "")[:500]

N_DOCS = 20_000
existing = collection.count()
print(f"Collection has {existing} documents.")
if existing == 0:
    print(f"Populating with {N_DOCS} items...")
    encoder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
    for i in tqdm(range(0, min(N_DOCS, len(train)), 1000)):
        batch = train[i : i + 1000]
        docs = [description(it) for it in batch]
        vecs = encoder.encode(docs).astype(float).tolist()
        metas = [{"category": getattr(it, "category", ""), "price": it.price} for it in batch]
        ids = [f"doc_{j}" for j in range(i, i + len(batch))]
        collection.add(ids=ids, documents=docs, embeddings=vecs, metadatas=metas)
    print(f"Done. Collection count: {collection.count()}")
else:
    print("Using existing vector store.")

## 4. SpecialistAgent (Modal fine-tuned pricer)

In [ ]:
from agents.specialist_agent import SpecialistAgent

specialist = SpecialistAgent()
sample = test[0]
est = specialist.price(description(sample))
print(f"Sample: {sample.title[:50]}...")
print(f"Actual: ${sample.price:.2f}, Specialist: ${est:.2f}")

## 5. FrontierAgent (RAG + frontier model)

In [ ]:
from agents.frontier_agent import FrontierAgent

frontier = FrontierAgent(collection)
est_f = frontier.price(description(sample))
print(f"Frontier estimate: ${est_f:.2f}")

## 6. Predictors for evaluator

Each takes an `Item` and returns a price (for `evaluate(predictor, test)`).

In [ ]:
def specialist_predictor(item):
    return specialist.price(description(item))

def frontier_predictor(item):
    return frontier.price(description(item))

def ensemble_predictor(item):
    s = specialist.price(description(item))
    f = frontier.price(description(item))
    return 0.5 * s + 0.5 * f

## 7. Run evaluation (Week 8 implementation)

Uses the same `evaluate()` as week8 day2: 200 items, report and charts. This **determines your result**.

In [ ]:
from agents.evaluator import evaluate

print("=== SpecialistAgent ===")
evaluate(specialist_predictor, test)

In [ ]:
print("=== FrontierAgent ===")
evaluate(frontier_predictor, test)

In [ ]:
print("=== Ensemble (Specialist + Frontier) ===")
evaluate(ensemble_predictor, test)

## 8. Optional: quick run with fewer items

Uncomment to test with 50 items and 3 workers.

In [ ]:
# evaluate(specialist_predictor, test, size=50, workers=3)

## 9. DealAgentFramework + Gradio UI

Uses the same framework as week8 day5: planning agent, scanner, messenger. The UI shows a table of deals and a button to run one planning cycle. Ensure `products_vectorstore` is in the current directory (we built it above).

In [ ]:
import gradio as gr
from deal_agent_framework import DealAgentFramework
from agents.deals import Opportunity, Deal

agent_framework = DealAgentFramework()
agent_framework.init_agents_as_needed()
print("DealAgentFramework ready.")

In [ ]:
def get_table(opps):
    if not opps:
        return [["No deals yet", "—", "—", "—", "—"]]
    return [
        [
            opp.deal.product_description[:80] + ("." if len(opp.deal.product_description) > 80 else ""),
            f"${opp.deal.price:.2f}",
            f"${opp.estimate:.2f}",
            f"${opp.discount:.2f}",
            opp.deal.url or "—",
        ]
        for opp in opps
    ]

def run_one_cycle():
    agent_framework.run()
    return get_table(agent_framework.memory)

with gr.Blocks(title="The Price is Right", fill_width=True) as ui:
    gr.Markdown("<div style='text-align: center; font-size: 24px'>The Price is Right — IbrahimSheriff</div>")
    gr.Markdown("Deals surfaced by the autonomous agent (Specialist + Frontier + Scanner + Planner).")
    with gr.Row():
        run_btn = gr.Button("Run one planning cycle")
    tbl = gr.Dataframe(
        headers=["Description", "Price", "Estimate", "Discount", "URL"],
        wrap=True,
        row_count=10,
        col_count=5,
        max_height=400,
    )
    run_btn.click(fn=run_one_cycle, outputs=[tbl])
    ui.load(fn=lambda: get_table(agent_framework.memory), outputs=[tbl])

ui.launch(inbrowser=True)